In [226]:
import pandas as  pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix
from imblearn.over_sampling import SMOTENC



In [227]:
df_orignal= pd.read_csv('train_data.csv')

In [228]:
df=df_orignal.copy()

In [229]:
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [230]:
len(df)

614

In [231]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


In [232]:
df.drop(['Loan_ID','Gender'],axis=1,inplace=True)

In [233]:
df.columns

Index(['Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status'],
      dtype='str')

In [234]:
df.ApplicantIncome.isnull().sum()

np.int64(0)

In [235]:
df.CoapplicantIncome.isnull().sum()

np.int64(0)

In [236]:
df['total_income'] = df['ApplicantIncome']+df['CoapplicantIncome']

In [237]:
df.head()

,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,total_income
0,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y,5849.0
1,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N,6091.0
2,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y,3000.0
3,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y,4941.0
4,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y,6000.0


In [238]:
df['LoanAmount'].isnull().sum()

np.int64(22)

In [239]:
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].mean())


In [240]:
df['LoanAmount'].isna().sum()

np.int64(0)

In [241]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,total_income
count,614.000000,614.000000,614.000000,600.00000,564.000000,614.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199,7024.705081
std,6109.041673,2926.248369,84.037468,65.12041,0.364878,6458.663872
min,150.000000,0.000000,9.000000,12.00000,0.000000,1442.000000
25%,2877.500000,0.000000,100.250000,360.00000,1.000000,4166.000000
50%,3812.500000,1188.500000,129.000000,360.00000,1.000000,5416.500000
75%,5795.000000,2297.250000,164.750000,360.00000,1.000000,7521.750000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000,81000.000000


In [242]:
df.nunique()

Married                2
Dependents             4
Education              2
Self_Employed          2
ApplicantIncome      505
CoapplicantIncome    287
LoanAmount           204
Loan_Amount_Term      10
Credit_History         2
Property_Area          3
Loan_Status            2
total_income         554
dtype: int64

In [243]:
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mean())
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])


In [244]:
df['Credit_History'].isna().sum()

np.int64(0)

In [245]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,total_income
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,5403.459283,1621.245798,146.412162,342.000000,0.855049,7024.705081
std,6109.041673,2926.248369,84.037468,64.372489,0.352339,6458.663872
min,150.000000,0.000000,9.000000,12.000000,0.000000,1442.000000
25%,2877.500000,0.000000,100.250000,360.000000,1.000000,4166.000000
50%,3812.500000,1188.500000,129.000000,360.000000,1.000000,5416.500000
75%,5795.000000,2297.250000,164.750000,360.000000,1.000000,7521.750000
max,81000.000000,41667.000000,700.000000,480.000000,1.000000,81000.000000


In [246]:
df.columns

Index(['Married', 'Dependents', 'Education', 'Self_Employed',
       'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status',
       'total_income'],
      dtype='str')

In [247]:
df.drop(['ApplicantIncome','CoapplicantIncome'],axis=1,inplace=True)

In [248]:
df.columns

Index(['Married', 'Dependents', 'Education', 'Self_Employed', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status',
       'total_income'],
      dtype='str')

In [249]:
df.head()

,Married,Dependents,Education,Self_Employed,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,total_income
0,No,0,Graduate,No,146.412162,360.0,1.0,Urban,Y,5849.0
1,Yes,1,Graduate,No,128.000000,360.0,1.0,Rural,N,6091.0
2,Yes,0,Graduate,Yes,66.000000,360.0,1.0,Urban,Y,3000.0
3,Yes,0,Not Graduate,No,120.000000,360.0,1.0,Urban,Y,4941.0
4,No,0,Graduate,No,141.000000,360.0,1.0,Urban,Y,6000.0


In [250]:
df['Monthly_Loan_Payment'] = (df['LoanAmount'] * 1000) / df['Loan_Amount_Term']
df['DTI_Ratio'] = df['Monthly_Loan_Payment'] / df['total_income']

In [251]:
df.head()

,Married,Dependents,Education,Self_Employed,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,total_income,Monthly_Loan_Payment,DTI_Ratio
0,No,0,Graduate,No,146.412162,360.0,1.0,Urban,Y,5849.0,406.700450,0.069533
1,Yes,1,Graduate,No,128.000000,360.0,1.0,Rural,N,6091.0,355.555556,0.058374
2,Yes,0,Graduate,Yes,66.000000,360.0,1.0,Urban,Y,3000.0,183.333333,0.061111
3,Yes,0,Not Graduate,No,120.000000,360.0,1.0,Urban,Y,4941.0,333.333333,0.067463
4,No,0,Graduate,No,141.000000,360.0,1.0,Urban,Y,6000.0,391.666667,0.065278


In [252]:
df.DTI_Ratio

0      0.069533
1      0.058374
2      0.061111
3      0.067463
4      0.065278
         ...   
609    0.068008
610    0.054121
611    0.084550
612    0.068501
613    0.080612
Name: DTI_Ratio, Length: 614, dtype: float64

In [253]:
df.drop(['Monthly_Loan_Payment'],axis=1,inplace=True)

In [254]:
df.head()

,Married,Dependents,Education,Self_Employed,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,total_income,DTI_Ratio
0,No,0,Graduate,No,146.412162,360.0,1.0,Urban,Y,5849.0,0.069533
1,Yes,1,Graduate,No,128.000000,360.0,1.0,Rural,N,6091.0,0.058374
2,Yes,0,Graduate,Yes,66.000000,360.0,1.0,Urban,Y,3000.0,0.061111
3,Yes,0,Not Graduate,No,120.000000,360.0,1.0,Urban,Y,4941.0,0.067463
4,No,0,Graduate,No,141.000000,360.0,1.0,Urban,Y,6000.0,0.065278


In [255]:
df.isnull().sum()

Married              3
Dependents          15
Education            0
Self_Employed       32
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
total_income         0
DTI_Ratio            0
dtype: int64

In [256]:
df.nunique()

Married               2
Dependents            4
Education             2
Self_Employed         2
LoanAmount          204
Loan_Amount_Term     11
Credit_History        2
Property_Area         3
Loan_Status           2
total_income        554
DTI_Ratio           611
dtype: int64

In [257]:
df['Dependents'].unique()

<StringArray>
['0', '1', '2', '3+', nan]
Length: 5, dtype: str

In [258]:
df['Married']= df['Married'].fillna(df['Married'].mode()[0])
df['Dependents']= df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed']= df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])

In [259]:
df.isnull().sum()

Married             0
Dependents          0
Education           0
Self_Employed       0
LoanAmount          0
Loan_Amount_Term    0
Credit_History      0
Property_Area       0
Loan_Status         0
total_income        0
DTI_Ratio           0
dtype: int64

In [260]:
df.drop_duplicates(inplace=True)

In [261]:
len(df)

613

In [262]:
df.columns

Index(['Married', 'Dependents', 'Education', 'Self_Employed', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status',
       'total_income', 'DTI_Ratio'],
      dtype='str')

In [263]:
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})


In [264]:
df['Loan_Status']

0      1
1      0
2      1
3      1
4      1
      ..
609    1
610    1
611    1
612    1
613    0
Name: Loan_Status, Length: 613, dtype: int64

In [265]:
df.head(3)

,Married,Dependents,Education,Self_Employed,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,total_income,DTI_Ratio
0,No,0,Graduate,No,146.412162,360.0,1.0,Urban,1,5849.0,0.069533
1,Yes,1,Graduate,No,128.000000,360.0,1.0,Rural,0,6091.0,0.058374
2,Yes,0,Graduate,Yes,66.000000,360.0,1.0,Urban,1,3000.0,0.061111


In [266]:
x = df.drop(columns=['Loan_Status'])   # Input features
y = df['Loan_Status']                  # Target

In [267]:
import pandas as pd

def separate_features(x):
    
    # Numerical features
    numerical_features = x.select_dtypes(include=['int64', 'float64']).columns.tolist()

    # Categorical features
    categorical_features = x.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

    return numerical_features, categorical_features

In [268]:
numerical_features, categorical_features = separate_features(x)

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'total_income', 'DTI_Ratio']

Categorical Features:
['Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area']


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2320\3188315646.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = x.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()


In [269]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [270]:
x_train.columns

Index(['Married', 'Dependents', 'Education', 'Self_Employed', 'LoanAmount',
       'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'total_income',
       'DTI_Ratio'],
      dtype='str')

In [271]:
x_train.shape , x_test.shape


((490, 10), (123, 10))

In [272]:
y_train.shape,y_test.shape

((490,), (123,))

In [273]:

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [274]:
preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [275]:
pipeline= Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])



In [276]:
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [3, 5, 7, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 5],
    'classifier__max_features': ['sqrt', 'log2']
}


In [277]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(x_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__max_depth': [3, 5, ...], 'classifier__max_features': ['sqrt', 'log2'], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more

In [278]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation Accuracy:")
print(grid_search.best_score_)

Best Parameters:
{'classifier__max_depth': 7, 'classifier__max_features': 'sqrt', 'classifier__min_samples_leaf': 5, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 300}

Best Cross Validation Accuracy:
0.8244897959183675


In [279]:
best_model = grid_search.best_estimator_

In [280]:
y_pred=best_model.predict(x_test)

In [281]:
accuracy_score(y_test,y_pred)

0.7886178861788617

In [282]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.84      0.41      0.55        39
           1       0.78      0.96      0.86        84

    accuracy                           0.79       123
   macro avg       0.81      0.69      0.71       123
weighted avg       0.80      0.79      0.76       123



In [283]:
print("\nTrain Accuracy:", best_model.score(x_train, y_train))
print("Test Accuracy:", best_model.score(x_test, y_test))



Train Accuracy: 0.8346938775510204
Test Accuracy: 0.7886178861788617


In [288]:
print(confusion_matrix(y_test,y_pred))

[[16 23]
 [ 3 81]]


In [290]:
best_model.feature_names_in_

array(['Married', 'Dependents', 'Education', 'Self_Employed',
       'LoanAmount', 'Loan_Amount_Term', 'Credit_History',
       'Property_Area', 'total_income', 'DTI_Ratio'], dtype=object)

In [291]:
import pandas as pd

new_customer = pd.DataFrame({
    'Married': ['Yes'],
    'Dependents': ['0'],
    'Education': ['Graduate'],
    'Self_Employed': ['No'],
    'LoanAmount': [150],
    'Loan_Amount_Term': [360],
    'Credit_History': [1],
    'Property_Area': ['Urban'],
    'total_income': [7000],
    'DTI_Ratio': [0.25]
})

prediction = best_model.predict(new_customer)

print("Loan Status:", prediction[0])

Loan Status: 1


In [ ]:
# import joblib
# joblib.dump(best_model,'loan_prediction_model.pkl')

['loan_prediction_model.pkl']